# 7차시 · 데이터 품질 검증 (Great Expectations)

이 노트북은 Iceberg `analytics` 기준 테이블 7개에 대해 **데이터 품질을 선언적으로 검증**합니다.

학습 목표
- 데이터 품질의 표준 차원(completeness · uniqueness · validity · consistency · freshness)을 이해한다.
- Great Expectations로 "규칙 목록 = 문서"가 되는 선언형 검증을 작성한다.
- 검증이 **없으면 조용히 통과하는** 오염을, 검증이 **있으면 잡아내는** 것을 직접 본다.

> 커널: **PySpark (DE5 Lakehouse)**. `spark` 세션과 `iceberg_lake`/`paimon_lake` 카탈로그가 자동 연결되어 있습니다.

## 0. 왜 데이터 품질 검증인가

5차시 Airflow DAG의 `validate_bi_metrics()`는 같은 검증을 파이썬 `assert`로 **코드 안에 묻어** 두었습니다.
- 문제: 어떤 규칙인지 알려면 코드를 읽어야 하고, 실패가 어느 규칙인지 한눈에 안 보인다.

7차시에서는 같은 의도를 **Great Expectations 선언형 스위트**로 승격합니다.
- 규칙 목록 자체가 문서이고, 통과/실패가 expectation 단위로 남는다.
- "DAG는 초록인데 데이터는 깨져 있다"(5차시 R5)를 **데이터 계층에서** 막는 게이트가 된다.

## 0-1. Spark 연결 확인

이 노트북은 **PySpark (DE5 Lakehouse)** 커널로 실행합니다. 그래서 `SparkSession`을 직접 만드는 코드가 없어도 `spark` 객체가 이미 준비되어 있어야 합니다.

아래 셀이 실패하면 먼저 커널이 `PySpark (DE5 Lakehouse)`인지, Docker stack과 Jupyter가 정상 실행 중인지 확인합니다.


In [ ]:
spark.sparkContext.setLogLevel("ERROR")

print("Spark", spark.version)
print("Spark app:", spark.sparkContext.appName)

spark.sql("SHOW CATALOGS").show(truncate=False)
spark.sql("SHOW TABLES IN iceberg_lake.analytics").show(truncate=False)


In [ ]:
import great_expectations as gx
from great_expectations import expectations as gxe
import pandas as pd

CATALOG, NS = "iceberg_lake", "analytics"
TABLES = [
    "olist_ux_events_clean", "olist_review_current", "olist_order_current",
    "olist_event_type_daily", "olist_funnel_daily", "olist_category_daily",
    "olist_review_sentiment_by_category",
]

print("Great Expectations", gx.__version__)
print("Spark", spark.version)
counts = {t: spark.table(f"{CATALOG}.{NS}.{t}").count() for t in TABLES}
pd.DataFrame({"table": list(counts), "rows": list(counts.values())})

## 1. Great Expectations 1.x 핵심 흐름

```
Context  →  DataSource  →  DataAsset  →  BatchDefinition  →  Batch
(작업공간)   (어디서)        (무엇을)       (어떻게 자를지)      (실제 데이터)

ExpectationSuite (= Expectation 목록)  ──validate──▶  ValidationResult
```

아래는 `olist_category_daily` 한 테이블로 이 흐름을 처음부터 작성한 예입니다.

In [ ]:
context = gx.get_context(mode="ephemeral")
df = spark.table(f"{CATALOG}.{NS}.olist_category_daily")

# 1) 어떤 데이터를 검증할지
ds = context.data_sources.add_spark(name="demo_spark")
asset = ds.add_dataframe_asset(name="category_daily")
batch_def = asset.add_batch_definition_whole_dataframe(name="whole")

# 2) 무엇을 만족해야 하는지 (Expectation 모음 = Suite)
suite = context.suites.add(gx.ExpectationSuite(name="category_daily_demo"))
suite.add_expectation(gxe.ExpectColumnValuesToNotBeNull(column="category_code"))
suite.add_expectation(gxe.ExpectColumnValuesToBeBetween(column="event_count", min_value=1))

# 3) 배치에 스위트 실행
batch = batch_def.get_batch(batch_parameters={"dataframe": df})
result = batch.validate(suite)
print("성공:", result.success)

In [ ]:
pd.DataFrame([
    {"expectation": r.expectation_config.type,
     "column": r.expectation_config.kwargs.get("column"),
     "success": r.success,
     "unexpected_count": (r.result or {}).get("unexpected_count")}
    for r in result.results
])

## 2. 표준 품질 차원 3가지 (GE로 선언)

| 차원 | 질문 | GE expectation 예 |
|---|---|---|
| **completeness** | 빠진 게 없는가 | `ExpectTableRowCountToEqual`, `ExpectColumnValuesToNotBeNull` |
| **uniqueness** | 키가 중복되지 않는가 | `ExpectColumnValuesToBeUnique` |
| **validity** | 값이 허용 범위/집합 안인가 | `ExpectColumnValuesToBeBetween`, `ExpectColumnValuesToBeInSet` |

7개 테이블 전체 규칙은 `labs/09-data-quality/data_quality_checks.py`에 정의되어 있습니다. 그 모듈을 그대로 재사용해 한 번에 검증합니다.

In [ ]:
import sys
sys.path.insert(0, "/workspace/labs/09-data-quality")
from data_quality_checks import expectations_for, EXPECTED_ROWS, run_ge_suite, summarize_result

context2 = gx.get_context(mode="ephemeral")
summary = []
for t in EXPECTED_ROWS:
    res = run_ge_suite(context2, spark, t)
    s = summarize_result(res)
    summary.append({"table": t,
                    "expectations": len(s),
                    "passed": sum(x["success"] for x in s),
                    "success": res.success})
pd.DataFrame(summary)

## 3. 의도적 실패 데모 — "검증이 없으면 조용히 통과한다"

실무 사례 N: *에러 허용치를 너무 높이면 실패 대신 데이터가 사라진다.* 잡이 죽지 않는다고 안전한 게 아닙니다.

여기서는 `olist_category_daily`의 **정상 데이터를 복사한 뒤 한 행만 오염**시켜(카테고리 NULL + event_count 0),
같은 스위트가 그 오염을 잡아내는지 봅니다. (실제 Iceberg 테이블은 건드리지 않습니다 — 메모리상 복사본만 오염)

In [ ]:
from pyspark.sql import functions as F

good = spark.table(f"{CATALOG}.{NS}.olist_category_daily")          # 759행, 정상
bad_row = (good.limit(1)
           .withColumn("category_code", F.lit(None).cast("string"))  # not-null 위반
           .withColumn("event_count", F.lit(0)))                     # between(>=1) 위반
bad = good.unionByName(bad_row)                                      # 760행 → row_count도 위반

ctx = gx.get_context(mode="ephemeral")
ds = ctx.data_sources.add_spark(name="bad_demo")
asset = ds.add_dataframe_asset(name="bad_category")
bd = asset.add_batch_definition_whole_dataframe(name="whole")
suite = ctx.suites.add(gx.ExpectationSuite(name="bad_category_suite"))
for e in expectations_for("olist_category_daily"):
    suite.add_expectation(e)

res = bd.get_batch(batch_parameters={"dataframe": bad}).validate(suite)
print("overall success:", res.success, "  ← False가 정상입니다 (DQ가 오염을 잡아냄)")
pd.DataFrame([
    {"expectation": r.expectation_config.type,
     "column": r.expectation_config.kwargs.get("column"),
     "success": r.success,
     "unexpected_count": (r.result or {}).get("unexpected_count"),
     "observed_value": (r.result or {}).get("observed_value")}
    for r in res.results
])

## 4. GE 기본에 없는 차원 — consistency · freshness

- **consistency**: 컬럼 간 관계(퍼널 단조성 `sessions >= purchase_sessions`, 집계 합 일치)는 단일 컬럼 expectation으로 표현하기 어렵다.
- **freshness**: "최신 데이터인가"는 시간 기준 비교라 GE 기본 expectation에 없다.

이런 규칙은 Spark로 직접 계산합니다 (모듈의 `run_custom_checks`).

In [ ]:
from data_quality_checks import run_custom_checks
pd.DataFrame(run_custom_checks(spark))

## 5. 정리 & 과제

- 표준 3차원(completeness/uniqueness/validity)은 GE로 **선언**, 관계/시간 규칙(consistency/freshness)은 **직접 계산**.
- 헤드리스 게이트로 실행: `./scripts/run-data-quality-checks.sh` (모두 통과 시 종료코드 0, 위반 시 1 → Airflow/CI 게이트로 사용 가능).

**과제 아이디어**
1. `olist_review_current`에 "rating이 null이 아니어야 한다" 규칙을 추가하고 통과시켜 보기.
2. 의도적 실패 데모를 `olist_funnel_daily`로 바꿔, 퍼널 단조성을 깨는 행을 만들어 custom check가 잡는지 확인.
3. 이 DQ 게이트를 5차시 Airflow DAG의 `validate_bi_metric_counts` **다음 task**로 넣는다면 어디에 두겠는가? 이유와 함께 적기.